In [1]:
import os 
# Get the current working directory
current_dir = os.getcwd()
print("Current working directory:", current_dir)

import pandas as pd
import scanpy as sc
import anndata as ad
from tqdm import tqdm
import matplotlib.pyplot as plt # import matplotlib to visualize our qc metrics
import subprocess
import sys
import seaborn as sns
import numpy as np
import cupy as cp
import cudf
import scipy.sparse as sp

# magic incantation to help matplotlib work with our jupyter notebook
%matplotlib inline 

sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')

sns.set(style="whitegrid")

Current working directory: /gpfs/commons/home/kisaev/Leaflet-analysis/tabula_sapien
scanpy==1.10.4 anndata==0.11.3 umap==0.5.6 numpy==1.26.4 scipy==1.14.1 pandas==2.2.3 scikit-learn==1.6.1 statsmodels==0.14.2 igraph==0.11.5 louvain==0.8.2 pynndescent==0.5.12


In [2]:
# Directory containing .h5ad files
adata_dir = "/gpfs/commons/datasets/controlled/CZI/tabula-sapiens/TabulaSapiens_v2/GeneExpressionMatrices"

# List all .h5ad files
adata_files = [os.path.join(adata_dir, f) for f in os.listdir(adata_dir) if f.endswith(".h5ad")]

# Collect all the data into a list 
adata_list = []

# Load all the .h5ad files
for adata_file in tqdm(adata_files):
    print(f"Reading {adata_file}")
    adata = sc.read_h5ad(adata_file)
    print(f"Number of cells and genes: {adata.n_obs}, {adata.n_vars}")
    adata_list.append(adata)

print(f"Loaded {len(adata_list)} datasets")

  0%|          | 0/29 [00:00<?, ?it/s]

Reading /gpfs/commons/datasets/controlled/CZI/tabula-sapiens/TabulaSapiens_v2/GeneExpressionMatrices/Large_Intestine_TSP1_30_version2d_10X_smartseq_scvi_Nov122024.h5ad


  3%|▎         | 1/29 [00:11<05:11, 11.11s/it]

Number of cells and genes: 30084, 61806
Reading /gpfs/commons/datasets/controlled/CZI/tabula-sapiens/TabulaSapiens_v2/GeneExpressionMatrices/Uterus_TSP1_30_version2d_10X_smartseq_scvi_Nov122024.h5ad


  7%|▋         | 2/29 [00:20<04:32, 10.08s/it]

...

Number of cells and genes: 48951, 61806
Reading /gpfs/commons/datasets/controlled/CZI/tabula-sapiens/TabulaSapiens_v2/GeneExpressionMatrices/Trachea_TSP1_30_version2d_10X_smartseq_scvi_Nov122024.h5ad


 97%|█████████▋| 28/29 [08:01<00:15, 15.19s/it]

Number of cells and genes: 22671, 61806
Reading /gpfs/commons/datasets/controlled/CZI/tabula-sapiens/TabulaSapiens_v2/GeneExpressionMatrices/Ear_TSP1_30_version2d_10X_smartseq_scvi_Nov122024.h5ad


100%|██████████| 29/29 [08:03<00:00, 16.68s/it]

Number of cells and genes: 3055, 61806
Loaded 29 datasets


In [3]:
# Let's first extract the adata.obs from each one and save those into one object
obs_list = [adata.obs for adata in adata_list]
obs = pd.concat(obs_list)

obs.head()

,donor,tissue,anatomical_position,method,cdna_plate,library_plate,notes,cdna_well,old_index,assay,...,total_counts_ercc,pct_counts_ercc,_scvi_batch,_scvi_labels,scvi_leiden_donorassay_full,age,sex,ethnicity,scvi_leiden_res05_tissue,sample_number
TSP14_LI_proximal_SS2_B134547_D101532_Epithelial_B20,TSP14,Large_Intestine,Proximal,smartseq,B134547,D101532,Epithelial,B20,TSP14_smartseq2_B134547_B20_D101532_B20_LI_pro...,SS2,...,0.0,0.000000,3,0,12,59,male,White,2,1
TSP14_LI_proximal_SS2_B134547_D101532_Epithelial_L10,TSP14,Large_Intestine,Proximal,smartseq,B134547,D101532,Epithelial,L10,TSP14_smartseq2_B134547_L10_D101532_L10_LI_pro...,SS2,...,465.0,0.131339,3,0,12,59,male,White,2,1
TSP14_LI_proximal_SS2_B134547_D101532_Epithelial_G16,TSP14,Large_Intestine,Proximal,smartseq,B134547,D101532,Epithelial,G16,TSP14_smartseq2_B134547_G16_D101532_G16_LI_pro...,SS2,...,778.0,0.083919,3,0,12,59,male,White,5,1
TSP14_LI_proximal_SS2_B134547_D101532_Epithelial_O19,TSP14,Large_Intestine,Proximal,smartseq,B134547,D101532,Epithelial,O19,TSP14_smartseq2_B134547_O19_D101532_O19_LI_pro...,SS2,...,487.0,0.024217,3,0,12,59,male,White,10,1
TSP14_LI_proximal_SS2_B134547_D101532_Epithelial_P21,TSP14,Large_Intestine,Proximal,smartseq,B134547,D101532,Epithelial,P21,TSP14_smartseq2_B134547_P21_D101532_P21_LI_pro...,SS2,...,1242.0,0.215953,3,0,12,59,male,White,2,1


In [4]:
adata_list[0].layers

Layers with keys: decontXcounts, log_normalized, raw_counts, scale_data

In [8]:
# 🚀 Step 1: Collect Gene Lists from All Datasets
gene_sets = [set(adata.var_names) for adata in adata_list]

# 🔥 Find Common Genes Across All Datasets
common_genes = sorted(set.intersection(*gene_sets))  # Sorting ensures consistent order
print(f"✅ Found {len(common_genes)} common genes across all datasets.")

# 🚀 Step 2: Align All Datasets to the Common Gene Set & Keep Only `SS2`
X_list = []   # Sparse expression matrices
obs_list = [] # Cell metadata
var_final = None  # Store final gene metadata

for adata in tqdm(adata_list):
    # 🚀 Subset only `SS2` cells
    ss2_cells = adata.obs["assay"] == "SS2"
    if ss2_cells.sum() == 0:
        continue  # Skip if no SS2 cells in this dataset

    adata = adata[ss2_cells, :].copy()  # Keep only SS2, make a copy to avoid view issues

    # 🚀 Replace `adata.X` with `raw_counts`
    if "raw_counts" in adata.layers:
        print(f"Using `adata.layers['raw_counts']` for {adata.shape[0]} cells")
        adata.X = adata.layers["raw_counts"].copy()  # Ensure raw counts are used
    else:
        print(f"⚠️ Skipping dataset with {adata.shape[0]} cells as `raw_counts` layer is missing")
        # print the layers available
        print(adata.obs.head())
        continue  # Skip this dataset if `raw_counts` is not available

    # 🚀 Subset and reorder genes to match the common gene set
    adata = adata[:, list(common_genes)].copy()  # **Reorders genes correctly!**
    print(f"Subsetted to {adata.shape[0]} cells and {adata.shape[1]} genes")

    # Ensure `X` is sparse
    if not sp.issparse(adata.X):
        adata.X = sp.csr_matrix(adata.X)

    X_list.append(adata.X)  # Store aligned sparse matrix
    obs_list.append(adata.obs.copy())  # Store cell metadata

    # Use the first dataset's `var` as the final gene metadata
    if var_final is None:
        var_final = adata.var.loc[list(common_genes)].copy()  # Keep gene metadata

# 🚀 Step 3: Merge Sparse Matrices & Metadata
print("Merging sparse matrices and metadata...")
X_merged = sp.vstack(X_list)  # Efficient sparse merging
obs_merged = pd.concat(obs_list, ignore_index=True)  # Faster `obs` concatenation

# 🚀 Step 4: Reconstruct the Final AnnData Object with RAW Counts
merged_adata = ad.AnnData(X=X_merged, obs=obs_merged, var=var_final)

✅ Found 61806 common genes across all datasets.


  0%|          | 0/29 [00:00<?, ?it/s]

Using `adata.layers['raw_counts']` for 1714 cells
Subsetted to 1714 cells and 61806 genes


  3%|▎         | 1/29 [00:00<00:06,  4.63it/s]

Using `adata.layers['raw_counts']` for 672 cells


...

 93%|█████████▎| 27/29 [00:04<00:00,  5.41it/s]

Using `adata.layers['raw_counts']` for 1898 cells
Subsetted to 1898 cells and 61806 genes
Using `adata.layers['raw_counts']` for 812 cells


100%|██████████| 29/29 [00:04<00:00,  6.34it/s]


Subsetted to 812 cells and 61806 genes
Merging sparse matrices and metadata...


/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [ ]:
# Save the Final Merged Dataset Efficiently
merged_adata.write_h5ad("/gpfs/commons/datasets/controlled/CZI/tabula-sapiens/TabulaSapiens_v2/GeneExpressionMatrices/merged_tabula_sapiens.h5ad", compression="gzip")

print(f"✅ Successfully merged {len(adata_list)} datasets! Final shape: {merged_adata.shape}")